In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../../..')

In [ ]:
is_hitter = True

In [ ]:
from Model.Constants import DATA_PREP_BINARY_FILE
from Model.Combined.DataPrep.Data_Prep import Combined_Data_Prep

data_prep = Combined_Data_Prep.Load_From_File("../../" + DATA_PREP_BINARY_FILE)

In [ ]:
io_list = data_prep.Generate_IO_Hitters(is_training=True) if is_hitter else data_prep.Generate_IO_Pitchers(is_training=True)

In [ ]:
from Model.Combined.Tuning.ProWar import objective, ProModelTuningRecipe
from functools import partial

recipe = ProModelTuningRecipe.INIT_HIDDEN
objective_func = partial(
    objective,
    io_list=io_list,
    data_prep=data_prep,
    is_hitter=is_hitter,
    recipe=recipe
)

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

study_name = "Pro_Hitter_WAR_Tuning" if is_hitter else "Pro_Pitcher_WAR_Tuning"
study_name += f"_{recipe.name}"
#optuna.delete_study(study_name=study_name, storage="sqlite:///Pro_Tune.db")
study = optuna.create_study(
    direction="minimize",
    load_if_exists=False,
    study_name=study_name,
    storage="sqlite:///Pro_Tune.db"
)

In [ ]:
study.optimize(objective_func, n_trials=50, show_progress_bar=True)

In [ ]:
print(f"Final best value after refinement: {study.best_value:.4f}")
print(f"Best hyperparameters: {study.best_params}")

In [ ]:
# Code to output a specific trial.  Given run/run randomness, typically a good late trial will be better
# Than one well before it that had better loss when repeated

# trial = study.get_trials()[196]
# print(trial.value)
# print(trial.params)

In [ ]:
import optuna.visualization as vis
vis.plot_param_importances(study).show()

In [ ]:
vis.plot_optimization_history(study).show()

In [ ]:
vis.plot_slice(study).show()